# E1b v3 — few-shot target-only

Mỗi lần chạy xử lý một cohort GEO. Feature selection, tuning và LR-L2 chỉ dùng target-train trong `fewshot_splits.csv`; target-test cố định và không được dùng ở bất kỳ bước nào trước đánh giá.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')

# One Kaggle version = one FS group across all four GEO target cohorts.
CLASSIFICATION_GROUP = 'filter'  # filter | embedded | wrappers
COHORTS = ['gse10245', 'gse30219', 'gse37745', 'gse42127']
K_GRID = [2, 3, 4, 5, 10]
METHODS_BY_GROUP = {
    'filter': ['f_test', 'mutual_info', 'mrmr', 'relieff'],
    'embedded': ['l1_lr', 'elastic_net', 'rf_importance', 'svm_rfe'],
    'wrappers': ['ga', 'bpso'],
}
assert CLASSIFICATION_GROUP in METHODS_BY_GROUP
METHODS = METHODS_BY_GROUP[CLASSIFICATION_GROUP]


In [ ]:
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
RUNTIME_IMPORTS = ('numpy', 'pandas', 'scipy', 'sklearn', 'tables', 'joblib', 'threadpoolctl')
try:
    for package in RUNTIME_IMPORTS:
        print(package, __import__(package).__version__)
except Exception as exc:
    raise RuntimeError('Restart the Kaggle session and rerun from Cell 1; do not install requirements.txt.') from exc
# ReliefF is the only E1b dependency absent from the standard Kaggle image.
# --no-deps preserves Kaggle's coherent NumPy/SciPy/scikit-learn stack.
try:
    import skrebate
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'skrebate==0.8.4'], check=True)
    import skrebate
print('skrebate installed')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    source, target = INPUT_V3_DATASET / folder, WORK_ROOT / folder
    assert source.is_dir(), f'Missing Kaggle input: {source}'
    if not target.exists():
        target.symlink_to(source, target_is_directory=True)
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('commit:', COMMIT)


In [ ]:
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.classify import evaluate_panel
from survey_v2.data import load_dataset, sha256, variance_prefilter
from survey_v2.methods import select
from scripts.build_v3_splits import FEWSHOT_N, validate_fewshot_split_manifest

def atomic_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def checkpoint(root, split, n_target, method):
    return root / 'checkpoints' / f's{split}_n{n_target}_{method}'

for COHORT in COHORTS:
    ds = load_dataset(WORK_ROOT, COHORT)
    manifest_path = ds.data_dir / 'fewshot_splits.csv'
    manifest = pd.read_csv(manifest_path, dtype={'n_target': str})
    errors = validate_fewshot_split_manifest(manifest, n_per_class=FEWSHOT_N[COHORT])
    assert not errors, errors
    index = {sample: i for i, sample in enumerate(ds.sample_ids)}
    output_root = WORK_ROOT / 'results' / 'survey_v3' / 'E1b_fewshot' / CLASSIFICATION_GROUP / COHORT
    for (split, n_target), frame in manifest.groupby(['split', 'n_target'], sort=True):
        train = np.array([index[s] for s in frame.loc[frame.role == 'train', 'sample_id']], dtype=int)
        test = np.array([index[s] for s in frame.loc[frame.role == 'test', 'sample_id']], dtype=int)
        assert set(ds.y[train]) == {0, 1} and set(ds.y[test]) == {0, 1}
        pre = variance_prefilter(ds.X[train], 5000)
        for method_id, method in enumerate(METHODS):
            base = checkpoint(output_root, split, n_target, method)
            metric_path, gene_path = base.with_suffix('.metrics.csv'), base.with_suffix('.genes.csv')
            if metric_path.is_file() and gene_path.is_file():
                continue
            seed = C.seed_for(C.dataset_seed_id(COHORT), int(frame.seed.iloc[0]), int(split), method_id)
            with threadpool_limits(1):
                chosen = select(method, ds.X[train][:, pre], ds.y[train], K_GRID, seed=seed, n_jobs=1)
            metrics, genes = [], []
            for k in K_GRID:
                columns = pre[chosen.panels[k]]
                assert len(columns) == k and len(set(columns.tolist())) == k
                with threadpool_limits(1):
                    result = evaluate_panel(ds.X[train][:, columns], ds.y[train], ds.X[test][:, columns], ds.y[test], classifier='lr_l2', seed=C.seed_for(seed, k), n_jobs=1)
                metrics.append({'cohort': COHORT, 'family': CLASSIFICATION_GROUP, 'strategy': 'target_only', 'split': split, 'seed': int(frame.seed.iloc[0]), 'n_target': n_target, 'method': method, 'k': k, 'n_train': len(train), 'n_test': len(test), 'auroc': result.auroc, 'bacc': result.bacc, 'mcc': result.mcc, 'threshold': result.threshold, 'hyperparam': result.hyperparam, 'n_selected': k, 'prefilter_p0': len(pre)})
                genes.extend({'cohort': COHORT, 'family': CLASSIFICATION_GROUP, 'strategy': 'target_only', 'split': split, 'n_target': n_target, 'method': method, 'k': k, 'rank': rank, 'gene': ds.genes[col]} for rank, col in enumerate(columns, 1))
            atomic_csv(pd.DataFrame(metrics), metric_path)
            atomic_csv(pd.DataFrame(genes), gene_path)
            print(f'{COHORT} {CLASSIFICATION_GROUP} split={split} n_target={n_target} {method}: done', flush=True)


In [ ]:
results_root = WORK_ROOT / 'results' / 'survey_v3' / 'E1b_fewshot' / CLASSIFICATION_GROUP
for COHORT in COHORTS:
    ds = load_dataset(WORK_ROOT, COHORT)
    manifest_path = ds.data_dir / 'fewshot_splits.csv'
    manifest = pd.read_csv(manifest_path, dtype={'n_target': str})
    output_root = results_root / COHORT
    metric_files = sorted((output_root / 'checkpoints').glob('*.metrics.csv'))
    gene_files = sorted((output_root / 'checkpoints').glob('*.genes.csv'))
    levels = sorted(manifest.n_target.astype(str).unique(), key=lambda x: (x == 'pool', int(x) if x != 'pool' else 10**9))
    expected = len(manifest.split.unique()) * len(levels) * len(METHODS)
    assert len(metric_files) == expected and len(gene_files) == expected, (COHORT, len(metric_files), len(gene_files), expected)
    atomic_csv(pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True), output_root / 'e1b_per_split.csv')
    atomic_csv(pd.concat([pd.read_csv(p) for p in gene_files], ignore_index=True), output_root / 'e1b_selected_genes.csv')
    config = {'experiment': 'E1b_fewshot_target_only_v3', 'cohort': COHORT, 'family': CLASSIFICATION_GROUP, 'cohorts_requested': COHORTS, 'strategy': 'target_only', 'methods': METHODS, 'k_grid': K_GRID, 'fewshot_levels': levels, 'fewshot_split_sha256': sha256(manifest_path), 'commit_hash': COMMIT, 'traceable': True}
    (output_root / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
archive = Path('/kaggle/working') / f'e1b_fewshot_{CLASSIFICATION_GROUP}_all_cohorts.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results/survey_v3/E1b_fewshot')
print('DONE:', archive)
